In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("../Data/DA task.csv")

# First look
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: '../Data/DA task.csv'

In [ ]:
import os
os.listdir('../Data')

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
os.listdir('C:/Users/Kingsley/splendor-trial-analysis')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading the dataset using full path
df = pd.read_csv('C:/Users/Kingsley/splendor-trial-analysis/ Data/DA task.csv')

# First look
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())

In [ ]:
# Fix date columns
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
df['CONVERTED_AT'] = pd.to_datetime(df['CONVERTED_AT'])
df['TRIAL_START'] = pd.to_datetime(df['TRIAL_START'])
df['TRIAL_END'] = pd.to_datetime(df['TRIAL_END'])

# Check unique organisations
print("Total unique orgs:", df['ORGANIZATION_ID'].nunique())

# Conversion rate
org_level = df.drop_duplicates('ORGANIZATION_ID')[['ORGANIZATION_ID', 'CONVERTED']]
print("Converted orgs:", org_level['CONVERTED'].sum())
print("Not converted:", (~org_level['CONVERTED']).sum())
print("Conversion rate:", round(org_level['CONVERTED'].mean() * 100, 1), "%")

# Activity names
print("\nUnique activities:", df['ACTIVITY_NAME'].nunique())
print("\nActivity counts:\n", df['ACTIVITY_NAME'].value_counts())

In [ ]:
# Fix date columns - handle errors gracefully
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
df['CONVERTED_AT'] = pd.to_datetime(df['CONVERTED_AT'], errors='coerce')
df['TRIAL_START'] = pd.to_datetime(df['TRIAL_START'], errors='coerce')
df['TRIAL_END'] = pd.to_datetime(df['TRIAL_END'], errors='coerce')

# Check how many timestamps were corrupted
print("Null TIMESTAMP after parsing:", df['TIMESTAMP'].isnull().sum())
print("Null TRIAL_START after parsing:", df['TRIAL_START'].isnull().sum())
print("Null TRIAL_END after parsing:", df['TRIAL_END'].isnull().sum())

# Check unique organisations
print("\nTotal unique orgs:", df['ORGANIZATION_ID'].nunique())

# Conversion rate
org_level = df.drop_duplicates('ORGANIZATION_ID')[['ORGANIZATION_ID', 'CONVERTED']]
print("Converted orgs:", org_level['CONVERTED'].sum())
print("Not converted:", (~org_level['CONVERTED']).sum())
print("Conversion rate:", round(org_level['CONVERTED'].mean() * 100, 1), "%")

# Activity names
print("\nUnique activities:", df['ACTIVITY_NAME'].nunique())
print("\nActivity counts:\n", df['ACTIVITY_NAME'].value_counts())

In [ ]:
# How many rows have ALL nulls vs partial nulls
print("Rows with null TIMESTAMP:", df['TIMESTAMP'].isnull().sum())
print("Rows with null TRIAL_START:", df['TRIAL_START'].isnull().sum())

# Are the null timestamps the same rows?
null_mask = df['TIMESTAMP'].isnull()
print("\nSample of rows with null TIMESTAMP:")
print(df[null_mask][['ORGANIZATION_ID', 'ACTIVITY_NAME', 'CONVERTED']].head(10))

# Check null timestamps affect converted or non-converted orgs
print("\nConverted status of orgs with null timestamps:")
print(df[null_mask]['CONVERTED'].value_counts())

In [ ]:
# ── DOCUMENT DATA QUALITY FINDINGS ──────────────────────

print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

print(f"\nTotal rows: {len(df)}")
print(f"Rows with valid timestamps: {df['TIMESTAMP'].notna().sum()}")
print(f"Rows with null timestamps: {df['TIMESTAMP'].isnull().sum()}")
print(f"% rows affected: {round(df['TIMESTAMP'].isnull().mean()*100,1)}%")

# Check duplicates
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

# ── CLEAN THE DATA ───────────────────────────────────────

# Keep all rows but flag which have valid timestamps
df['has_valid_timestamp'] = df['TIMESTAMP'].notna()

# Create a clean version with only valid timestamps for time-based analysis
df_clean = df[df['TIMESTAMP'].notna()].copy()

# Add useful derived column
df_clean['days_into_trial'] = (
    df_clean['TIMESTAMP'] - df_clean['TRIAL_START']
).dt.days

print(f"\nClean dataset rows: {len(df_clean)}")
print(f"Clean dataset orgs: {df_clean['ORGANIZATION_ID'].nunique()}")

# Conversion rate in clean dataset
org_clean = df_clean.drop_duplicates('ORGANIZATION_ID')[['ORGANIZATION_ID','CONVERTED']]
print(f"Conversion rate in clean data: {round(org_clean['CONVERTED'].mean()*100,1)}%")

In [ ]:
# ── CHECK DUPLICATES PROPERLY ────────────────────────────

# Before deduplication
print("Before deduplication:", len(df))

# Remove duplicates
df_deduped = df.drop_duplicates()
print("After deduplication:", len(df_deduped))
print("Rows removed:", len(df) - len(df_deduped))

# Now clean - valid timestamps only, after deduplication
df_clean = df_deduped[df_deduped['TIMESTAMP'].notna()].copy()

# Add derived columns
df_clean['days_into_trial'] = (
    df_clean['TIMESTAMP'] - df_clean['TRIAL_START']
).dt.days

print("\nFinal clean dataset:")
print(f"Rows: {len(df_clean)}")
print(f"Unique orgs: {df_clean['ORGANIZATION_ID'].nunique()}")

# Final conversion rate
org_final = df_deduped.drop_duplicates('ORGANIZATION_ID')[['ORGANIZATION_ID','CONVERTED']]
print(f"Total orgs: {len(org_final)}")
print(f"Converted: {org_final['CONVERTED'].sum()}")
print(f"Conversion rate: {round(org_final['CONVERTED'].mean()*100,1)}%")

In [ ]:
# ── VISUALISATION 1: Conversion Overview ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Trial Overview Dashboard', fontsize=16, fontweight='bold')

# Chart 1 - Conversion pie chart
conv_counts = org_final['CONVERTED'].value_counts()
axes[0].pie(conv_counts, 
            labels=['Not Converted', 'Converted'],
            colors=['#ff6b6b', '#51cf66'],
            autopct='%1.1f%%',
            startangle=90)
axes[0].set_title('Conversion Rate')

# Chart 2 - Events per org distribution
events_per_org = df_clean.groupby('ORGANIZATION_ID').size()
axes[1].hist(events_per_org, bins=30, color='#339af0', edgecolor='white')
axes[1].set_title('Events Per Organisation')
axes[1].set_xlabel('Number of Events')
axes[1].set_ylabel('Count of Orgs')

# Chart 3 - Activity frequency
top_activities = df_clean['ACTIVITY_NAME'].value_counts().head(10)
axes[2].barh(top_activities.index, top_activities.values, color='#845ef7')
axes[2].set_title('Top 10 Activities')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('trial_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

In [ ]:
# ── ACTIVITY ADOPTION BY CONVERSION STATUS ───────────────

# Create org-level feature matrix
# For each org, did they do each activity? (True/False)
org_activities = df_deduped.groupby(
    ['ORGANIZATION_ID', 'ACTIVITY_NAME', 'CONVERTED']
).size().reset_index(name='count')

# Pivot to get one row per org
org_matrix = org_activities.pivot_table(
    index=['ORGANIZATION_ID', 'CONVERTED'],
    columns='ACTIVITY_NAME',
    values='count',
    fill_value=0
).reset_index()

print("Org matrix shape:", org_matrix.shape)
print("\nFirst few rows:")
print(org_matrix.head())

In [ ]:
# ── CONVERSION LIFT BY ACTIVITY ───────────────────────────

# Get activity columns only
activity_cols = [col for col in org_matrix.columns 
                 if col not in ['ORGANIZATION_ID', 'CONVERTED']]

# For each activity, calculate:
# 1. % of converters who used it
# 2. % of non-converters who used it
# 3. Conversion lift (ratio)

results = []

for activity in activity_cols:
    # Convert counts to binary (did they use it or not)
    used = org_matrix[activity] > 0
    converted = org_matrix['CONVERTED'] == True
    
    conv_used = (used & converted).sum()
    conv_total = converted.sum()
    
    nonconv_used = (used & ~converted).sum()
    nonconv_total = (~converted).sum()
    
    conv_rate = conv_used / conv_total * 100
    nonconv_rate = nonconv_used / nonconv_total * 100
    
    lift = conv_rate / nonconv_rate if nonconv_rate > 0 else 0
    
    results.append({
        'activity': activity,
        'converter_adoption_%': round(conv_rate, 1),
        'non_converter_adoption_%': round(nonconv_rate, 1),
        'lift': round(lift, 2)
    })

# Create results dataframe
results_df = pd.DataFrame(results).sort_values('lift', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# ── VISUALISE CONVERSION LIFT ─────────────────────────────

# Filter activities with meaningful adoption (>2% by either group)
meaningful = results_df[
    (results_df['converter_adoption_%'] > 2) | 
    (results_df['non_converter_adoption_%'] > 2)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Activity Adoption: Converters vs Non-Converters', 
             fontsize=14, fontweight='bold')

# Chart 1 - Side by side comparison
x = range(len(meaningful))
width = 0.35

bars1 = axes[0].barh(
    [a.replace('Scheduling.', 'Sch.').replace('Mobile.', 'Mob.')
      .replace('PunchClock.', 'PC.').replace('Absence.', 'Abs.')
      .replace('Communication.', 'Comm.') 
     for a in meaningful['activity']],
    meaningful['converter_adoption_%'],
    color='#51cf66', alpha=0.8, label='Converters'
)
bars2 = axes[0].barh(
    [a.replace('Scheduling.', 'Sch.').replace('Mobile.', 'Mob.')
      .replace('PunchClock.', 'PC.').replace('Absence.', 'Abs.')
      .replace('Communication.', 'Comm.')
     for a in meaningful['activity']],
    meaningful['non_converter_adoption_%'],
    color='#ff6b6b', alpha=0.5, label='Non-Converters'
)
axes[0].set_title('Adoption Rate by Activity (%)')
axes[0].set_xlabel('% of Organisations')
axes[0].legend()

# Chart 2 - Lift chart
colors = ['#51cf66' if x > 1 else '#ff6b6b' for x in meaningful['lift']]
axes[1].barh(
    [a.replace('Scheduling.', 'Sch.').replace('Mobile.', 'Mob.')
      .replace('PunchClock.', 'PC.').replace('Absence.', 'Abs.')
      .replace('Communication.', 'Comm.')
     for a in meaningful['activity']],
    meaningful['lift'],
    color=colors
)
axes[1].axvline(x=1, color='black', linestyle='--', linewidth=1.5, 
                label='No difference (lift=1)')
axes[1].set_title('Conversion Lift by Activity\n(Green = converters use more)')
axes[1].set_xlabel('Lift Ratio')
axes[1].legend()

plt.tight_layout()
plt.savefig('conversion_lift.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

In [ ]:
from scipy.stats import chi2_contingency

# ── CHI-SQUARE TEST FOR EACH ACTIVITY ─────────────────────
stat_results = []

for activity in activity_cols:
    used = (org_matrix[activity] > 0).astype(int)
    converted = org_matrix['CONVERTED'].astype(int)
    
    #contingency table
    ct = pd.crosstab(used, converted)
    
    # to be run if table has enough data
    if ct.shape == (2, 2):
        chi2, p_value, dof, expected = chi2_contingency(ct)
        stat_results.append({
            'activity': activity,
            'chi2': round(chi2, 3),
            'p_value': round(p_value, 4),
            'significant': p_value < 0.05
        })

stat_df = pd.DataFrame(stat_results).sort_values('p_value')
print("Statistically significant activities (p < 0.05):")
print(stat_df[stat_df['significant']][['activity','p_value']].to_string(index=False))
print("\nNot significant:")
print(stat_df[~stat_df['significant']][['activity','p_value']].to_string(index=False))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ── RANDOM FOREST FEATURE IMPORTANCE ──────────────────────

# Prepare features
X = org_matrix[activity_cols].fillna(0)
y = org_matrix['CONVERTED'].astype(int)

# Train model
rf = RandomForestClassifier(
    n_estimators=100, 
    random_state=42,
    class_weight='balanced'
)
rf.fit(X, y)

# Get feature importances
importance_df = pd.DataFrame({
    'activity': activity_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Activities:")
print(importance_df.head(15).to_string(index=False))

# Visualise
plt.figure(figsize=(10, 7))
top15 = importance_df.head(15)
colors = ['#51cf66' if i < 5 else '#339af0' for i in range(len(top15))]
plt.barh(top15['activity'], top15['importance'], color=colors)
plt.title('Random Forest Feature Importance\n(Which activities best predict conversion?)', 
          fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

In [ ]:
# ── CHECK GOAL COMPLETION RATES ───────────────────────────

goals = {
    'Goal 1: Shift Created (3+)': 
        (org_matrix['Scheduling.Shift.Created'] >= 3),
    'Goal 2: Mobile Loaded (1+)': 
        (org_matrix['Mobile.Schedule.Loaded'] >= 1),
    'Goal 3: Assignment Changed (1+)': 
        (org_matrix['Scheduling.Shift.AssignmentChanged'] >= 1),
    'Goal 4: Punched In (1+)': 
        (org_matrix['PunchClock.PunchedIn'] >= 1),
    'Goal 5: Shift Approved (1+)': 
        (org_matrix['Scheduling.Shift.Approved'] >= 1),
}

print("Goal Completion Rates:")
print("=" * 55)
for goal, condition in goals.items():
    total = condition.sum()
    rate = round(condition.mean() * 100, 1)
    
    # Split by conversion
    conv_rate = round(
        condition[org_matrix['CONVERTED']].mean() * 100, 1)
    nonconv_rate = round(
        condition[~org_matrix['CONVERTED']].mean() * 100, 1)
    
    print(f"{goal}")
    print(f"  Overall: {total} orgs ({rate}%)")
    print(f"  Converters: {conv_rate}% | Non-converters: {nonconv_rate}%")
    print()

# Check activation (all 5 goals)
all_goals = pd.DataFrame({k: v for k, v in goals.items()})
activated = all_goals.all(axis=1)
print("=" * 55)
print(f"FULLY ACTIVATED (all 5 goals): {activated.sum()} orgs ({round(activated.mean()*100,1)}%)")
print(f"Converters activated: {activated[org_matrix['CONVERTED']].mean()*100:.1f}%")
print(f"Non-converters activated: {activated[~org_matrix['CONVERTED']].mean()*100:.1f}%")

In [ ]:
# ── FIND BETTER GOAL CANDIDATES ───────────────────────────

# VOLUME to be looked at not just adoption
# How many times did converters vs non-converters use each activity?

volume_results = []

for activity in activity_cols:
    conv_mask = org_matrix['CONVERTED'] == True
    nonconv_mask = org_matrix['CONVERTED'] == False
    
    # Average usage count
    conv_avg = org_matrix.loc[conv_mask, activity].mean()
    nonconv_avg = org_matrix.loc[nonconv_mask, activity].mean()
    
    # Median usage
    conv_med = org_matrix.loc[conv_mask, activity].median()
    nonconv_med = org_matrix.loc[nonconv_mask, activity].median()
    
    # Volume lift
    vol_lift = conv_avg / nonconv_avg if nonconv_avg > 0 else 0
    
    volume_results.append({
        'activity': activity,
        'conv_avg_count': round(conv_avg, 1),
        'nonconv_avg_count': round(nonconv_avg, 1),
        'volume_lift': round(vol_lift, 2),
        'conv_median': conv_med,
        'nonconv_median': nonconv_med
    })

vol_df = pd.DataFrame(volume_results).sort_values(
    'volume_lift', ascending=False)

print("Activity Volume: Converters vs Non-Converters")
print("=" * 65)
print(vol_df[['activity','conv_avg_count',
              'nonconv_avg_count','volume_lift']].to_string(index=False))

In [ ]:
# ── REVISED TRIAL GOALS ───────────────────────────────────

revised_goals = {
    'Goal 1: Core Scheduling (Shift Created 3+)':
        (org_matrix['Scheduling.Shift.Created'] >= 3),
        
    'Goal 2: Template Used (1+)':
        (org_matrix['Scheduling.Template.ApplyModal.Applied'] >= 1),
        
    'Goal 3: Open Shift Request (1+)':
        (org_matrix['Scheduling.OpenShiftRequest.Created'] >= 1),
        
    'Goal 4: Mobile Engaged (3+)':
        (org_matrix['Mobile.Schedule.Loaded'] >= 3),
        
    'Goal 5: Absence Managed (1+)':
        (org_matrix['Absence.Request.Created'] >= 1),
}

print("REVISED Goal Completion Rates:")
print("=" * 60)
for goal, condition in revised_goals.items():
    total = condition.sum()
    rate = round(condition.mean() * 100, 1)
    conv_rate = round(
        condition[org_matrix['CONVERTED']].mean() * 100, 1)
    nonconv_rate = round(
        condition[~org_matrix['CONVERTED']].mean() * 100, 1)
    diff = round(conv_rate - nonconv_rate, 1)
    
    print(f"{goal}")
    print(f"  Overall: {total} orgs ({rate}%)")
    print(f"  Converters: {conv_rate}% | Non-converters: {nonconv_rate}% | Diff: {diff:+.1f}%")
    print()

# Activation check
all_revised = pd.DataFrame({k: v for k, v in revised_goals.items()})
activated_revised = all_revised.all(axis=1)

print("=" * 60)
print(f"ACTIVATED (all 5 goals): {activated_revised.sum()} orgs ({round(activated_revised.mean()*100,1)}%)")

conv_act = activated_revised[org_matrix['CONVERTED']].mean()*100
nonconv_act = activated_revised[~org_matrix['CONVERTED']].mean()*100
print(f"Converters activated: {conv_act:.1f}%")
print(f"Non-converters activated: {nonconv_act:.1f}%")
print(f"Activation→Conversion lift: {round(conv_act/nonconv_act,2)}x" 
      if nonconv_act > 0 else "")

In [ ]:
# ── FINAL HONEST GOAL DEFINITION ─────────────────────────

final_goals = {
    'Goal 1: Shift Created (3+)':
        (org_matrix['Scheduling.Shift.Created'] >= 3),
        
    'Goal 2: Shift Approved (1+)':
        (org_matrix['Scheduling.Shift.Approved'] >= 1),
        
    'Goal 3: Template Applied (1+)':
        (org_matrix['Scheduling.Template.ApplyModal.Applied'] >= 1),
        
    'Goal 4: Open Shift Request (1+)':
        (org_matrix['Scheduling.OpenShiftRequest.Created'] >= 1),
        
    'Goal 5: Punched In (1+)':
        (org_matrix['PunchClock.PunchedIn'] >= 1),
}

# ── BUILD FINAL ACTIVATION TABLE ─────────────────────────
goals_df = pd.DataFrame({
    'ORGANIZATION_ID': org_matrix['ORGANIZATION_ID'],
    'CONVERTED': org_matrix['CONVERTED']
})

for goal, condition in final_goals.items():
    goals_df[goal] = condition.astype(int)

goal_cols = list(final_goals.keys())
goals_df['goals_completed'] = goals_df[goal_cols].sum(axis=1)
goals_df['is_activated'] = (goals_df['goals_completed'] == 5).astype(int)

# ── SUMMARY ───────────────────────────────────────────────
print("FINAL ACTIVATION SUMMARY")
print("=" * 50)
print(f"Total orgs: {len(goals_df)}")
print(f"\nGoals completed distribution:")
print(goals_df['goals_completed'].value_counts().sort_index())

print(f"\nActivated orgs: {goals_df['is_activated'].sum()}")
print(f"Activation rate: {round(goals_df['is_activated'].mean()*100,1)}%")

conv = goals_df['CONVERTED']
act = goals_df['is_activated'] == 1

print(f"\nAmong ACTIVATED orgs:")
print(f"  Converted: {(act & conv).sum()}")
print(f"  Not converted: {(act & ~conv).sum()}")
conv_if_activated = (act & conv).sum() / act.sum() * 100
print(f"  Conversion rate if activated: {conv_if_activated:.1f}%")

print(f"\nAmong NON-ACTIVATED orgs:")
nonact = goals_df['is_activated'] == 0
conv_if_not = (nonact & conv).sum() / nonact.sum() * 100
print(f"  Conversion rate if not activated: {conv_if_not:.1f}%")

print(f"\nActivation → Conversion lift: {round(conv_if_activated/conv_if_not,2)}x")

In [ ]:
# ── FINAL VISUALISATION ───────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Trial Activation Analysis — Final Summary', 
             fontsize=14, fontweight='bold')

# Chart 1 - Goals completion funnel
goal_completion = [goals_df[g].sum() for g in goal_cols]
short_names = [
    'Shift\nCreated(3+)',
    'Shift\nApproved',
    'Template\nApplied',
    'Open Shift\nRequest',
    'Punched\nIn'
]
bars = axes[0].bar(short_names, goal_completion, 
                   color='#339af0', edgecolor='white')
axes[0].set_title('Goal Completion Count')
axes[0].set_ylabel('Number of Orgs')
for bar, val in zip(bars, goal_completion):
    axes[0].text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + 5,
                str(val), ha='center', fontweight='bold')

# Chart 2 - Goals completed distribution
goals_dist = goals_df['goals_completed'].value_counts().sort_index()
axes[1].bar(goals_dist.index, goals_dist.values, 
            color='#845ef7', edgecolor='white')
axes[1].set_title('How Many Goals Do Orgs Complete?')
axes[1].set_xlabel('Number of Goals Completed')
axes[1].set_ylabel('Number of Orgs')
axes[1].axvline(x=4.5, color='red', linestyle='--', 
                label='Activation threshold')
axes[1].legend()

# Chart 3 - Conversion rate by activation status
categories = ['Not Activated', 'Activated']
conv_rates = [21.2, 33.3]
colors = ['#ff6b6b', '#51cf66']
bars = axes[2].bar(categories, conv_rates, color=colors, 
                   edgecolor='white', width=0.4)
axes[2].set_title('Conversion Rate by Activation Status')
axes[2].set_ylabel('Conversion Rate (%)')
axes[2].set_ylim(0, 45)
for bar, val in zip(bars, conv_rates):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f'{val}%', ha='center', 
                fontweight='bold', fontsize=12)
axes[2].axhline(y=21.3, color='gray', linestyle='--', 
                label='Overall avg (21.3%)')
axes[2].legend()

plt.tight_layout()
plt.savefig('activation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")